# WISER Module 5 Colab Template: Data Visualization and Communication

        **Learning objective:** Build an accessible, honest, stigma-free visualization and one-page brief for a public health decision-maker.

> **Before you begin:** In Google Colab, choose **File → Save a copy in Drive**. Work only in your copy.
>
> This notebook connects to your prior WISER work. If you completed earlier modules, you may import outputs from those notebooks where prompted.
>
> **Privacy reminder:** Do not upload protected health information, identifiable clinical notes, or non-approved institutional data. Use the WISER synthetic datasets unless your instructor has explicitly approved another dataset.

In [ ]:
MODULE_ID = "Module 5"
NOTEBOOK_TITLE = "WISER Module 5: Data Visualization and Communication"

## Setup

In [ ]:
from pathlib import Path
import json
import textwrap
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PALETTE = {
    "teal": "#20808D",
    "rust": "#A84B2F",
    "dark_teal": "#1B474D",
    "light_cyan": "#BCE2E7",
    "mauve": "#944454",
    "gold": "#FFC553",
}

STIGMA_TERMS = {
    "addict": "person with a substance use disorder",
    "addicts": "people with substance use disorders",
    "abuser": "person with a substance use disorder",
    "drug abuse": "substance use or substance use disorder",
    "clean": "negative toxicology / not currently using",
    "dirty": "positive toxicology",
    "junkie": "person with a substance use disorder",
}

def stigma_audit(text):
    '''Return potentially stigmatizing terms found in a string.'''
    text_lower = str(text).lower()
    return {term: suggestion for term, suggestion in STIGMA_TERMS.items() if term in text_lower}

def safe_label(text):
    '''Lightweight learner-facing label checker. Human review is still required.'''
    hits = stigma_audit(text)
    if hits:
        print("Review this wording:")
        for term, suggestion in hits.items():
            print(f" - Replace '{term}' with '{suggestion}' when clinically appropriate.")
    return text

def quick_profile(df, name="dataset"):
    '''Basic data profile for WISER learner notebooks.'''
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(1),
        "unique_n": df.nunique(dropna=True),
    })
    display(profile)
    display(df.head())

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 150, "axes.titleweight": "bold"})

## Section 1: Data Loading and Exploration

In [ ]:
DATA_URL = ""  # TODO: paste final WISER overdose visualization CSV URL, if available

if DATA_URL:
    overdose = pd.read_csv(DATA_URL)
else:
    months = pd.date_range("2019-01-01", "2022-12-01", freq="MS")
    overdose = pd.DataFrame({
        "month": np.tile(months, 2),
        "geography": ["County"] * len(months) + ["State"] * len(months),
    })
    base = np.linspace(10, 18, len(months)) + np.random.normal(0, 1.2, len(months))
    overdose["rate_per_100k"] = np.concatenate([base * 1.25, base])
    overdose["provisional"] = overdose["month"] >= "2022-07-01"
    overdose["drug_class"] = "synthetic opioids"

quick_profile(overdose, "overdose_visualization_data")

## Section 2: Chart Choice and Audience

In [ ]:
audience = "county commissioner"
decision = "allocate supplemental naloxone and low-barrier treatment funds"
chart_question = "Did monthly overdose mortality rise faster locally than statewide?"

print("Audience:", audience)
print("Decision:", decision)
print("Chart question:", chart_question)

## Section 3: Build the Primary Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for geo, g in overdose.groupby("geography"):
    ax.plot(g["month"], g["rate_per_100k"], marker="o", linewidth=2, label=geo)

prov_start = overdose.loc[overdose["provisional"], "month"].min()
if pd.notna(prov_start):
    ax.axvspan(prov_start, overdose["month"].max(), color="#FFC553", alpha=0.18, label="Provisional period")

ax.set_title("County overdose mortality rose faster than the state comparator\\nMonthly rate per 100,000; provisional months shaded", loc="left")
ax.set_xlabel("")
ax.set_ylabel("Deaths per 100,000")
ax.legend(frameon=False)

note = "Note: Synthetic template data shown. Replace with CDC WONDER, provisional overdose, SUDORS, or instructor-approved data for final submission."
fig.text(0.01, -0.05, textwrap.fill(note, 120), ha="left", va="top", fontsize=8)
plt.tight_layout()

chart_path = OUTPUT_DIR / "module5_primary_chart.png"
fig.savefig(chart_path, bbox_inches="tight")
plt.show()
print("Saved:", chart_path)

### Alt Text Draft

Write alt text that describes the key finding, not just the chart type.

In [ ]:
alt_text = "Line chart showing a higher synthetic monthly overdose mortality rate for the county than the state comparator, with recent provisional months shaded."
print(alt_text)

## Section 4: Stigma-Free Label Audit

In [ ]:
labels_to_check = [
    "Monthly overdose deaths involving synthetic opioids",
    "People with substance use disorder",
    "Positive toxicology result",
]
for label in labels_to_check:
    print("\\nLabel:", label)
    safe_label(label)

## Section 5: Foundational Pathway Brief

In [ ]:
brief = {
    "question": chart_question,
    "headline": "Synthetic example: the county rate increased faster than the state comparator during the shaded provisional period.",
    "caveats": [
        "The final months are provisional.",
        "Synthetic data are used in this template.",
        "Rates require accurate population denominators."
    ],
    "ask": "Specify the funding decision, amount, and accountability period."
}
display(pd.Series(brief))

## Section 6: Applied Pathway Extension

In [ ]:
# Applied task ideas:
# 1. Replace synthetic data with a CDC WONDER export.
# 2. Add an SVI or rural/urban equity overlay.
# 3. Export a one-page PDF brief.
print("Applied extension: replace DATA_URL or upload a CDC WONDER export, then rerun all cells.")

## Section 7: FAIR Export and Submission

In [ ]:
metadata = {
    "module": MODULE_ID,
    "notebook": NOTEBOOK_TITLE,
    "created_by": "WISER learner",
    "uses_synthetic_or_public_data": True,
    "contains_phi": False,
    "fair_notes": {
        "findable": "Outputs saved with clear names in outputs/.",
        "accessible": "Notebook and exported files can be shared through the course portal.",
        "interoperable": "CSV/JSON/Markdown formats used where possible.",
        "reusable": "README and metadata describe inputs, assumptions, and outputs.",
    },
}

(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
(OUTPUT_DIR / "README.md").write_text(f'''# {NOTEBOOK_TITLE} Outputs

This folder contains learner-generated outputs for {MODULE_ID}.

## Files

- `metadata.json`: FAIR-oriented metadata
- Additional module-specific artifacts produced by notebook cells

## Privacy

This template is designed for synthetic or public data. Do not include PHI or identifiable institutional data.
''')

print("FAIR export complete.")
print("Generated:", sorted(str(p) for p in OUTPUT_DIR.iterdir()))